# Notebook 03a — Non-IID Experiments on CIFAR-10

This notebook evaluates the impact of Non-IID data partitioning on membership
inference attack success for **CIFAR-10 only**.

**What this notebook does:**
- Runs FL + MIA pipeline under IID partitioning (baseline)
- Runs FL + MIA pipeline under Non-IID partitioning (5 classes per client)
- Compares attack AUC between IID and Non-IID conditions
- Produces comparison plots and saves results

**Key question:** Does Non-IID data increase privacy leakage on CIFAR-10?

**Configuration:** 5 clients | 10 rounds | 3 local epochs
This notebook is **fully self-contained** — no file uploads needed.


## 1. Imports & Configuration


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import json, os, gc

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    roc_auc_score, roc_curve
)
import matplotlib.pyplot as plt

NUM_CLIENTS          = 5
NUM_ROUNDS           = 10
LOCAL_EPOCHS         = 3
BATCH_SIZE           = 64
LEARNING_RATE        = 0.001
NUM_CLASSES          = 10
RANDOM_SEED          = 42
CLASSES_PER_CLIENT   = 5   # each client gets 5 out of 10 classes

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
gc.collect()
tf.keras.backend.clear_session()

OUTPUT_DIR = '/content/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'TensorFlow : {tf.__version__}')
print(f'GPU        : {tf.config.list_physical_devices("GPU")}')
print(f'Config     : {NUM_CLIENTS} clients | {NUM_ROUNDS} rounds | {LOCAL_EPOCHS} local epochs')
print(f'Non-IID    : {CLASSES_PER_CLIENT} classes per client')


## 2. Load CIFAR-10


In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0
y_train = y_train.flatten()
y_test  = y_test.flatten()

print(f'Training set : {x_train.shape}')
print(f'Test set     : {x_test.shape}')
print(f'Classes      : {len(np.unique(y_train))}')


## 3. Partitioning & Helper Functions


In [ ]:
def partition_data_iid(x, y, num_clients, seed=42):
    """IID: shuffle and split equally across clients."""
    rng     = np.random.default_rng(seed)
    indices = rng.permutation(len(x))
    splits  = np.array_split(indices, num_clients)
    client_data    = [(x[idx], y[idx]) for idx in splits]
    client_indices = [idx.tolist() for idx in splits]
    return client_data, client_indices


def partition_data_noniid(x, y, num_clients, classes_per_client=5, seed=42):
    """
    Non-IID: each client receives data from a limited set of classes.
    Classes are assigned cyclically across clients.
    """
    num_classes = len(np.unique(y))
    rng = np.random.default_rng(seed)
    class_indices = {c: rng.permutation(np.where(y == c)[0]) for c in range(num_classes)}

    client_data    = []
    client_indices = []

    for i in range(num_clients):
        assigned = [(i * classes_per_client + j) % num_classes for j in range(classes_per_client)]
        idx = np.concatenate([class_indices[c] for c in assigned])
        client_data.append((x[idx], y[idx]))
        client_indices.append(idx.tolist())
        print(f'  Client {i+1}: {len(idx):>6} samples | classes: {assigned}')

    return client_data, client_indices


def build_model(num_classes=10, learning_rate=0.001):
    model = keras.Sequential([
        layers.Input(shape=(32, 32, 3)),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax'),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


def fedavg_aggregate(client_weights, client_sizes):
    total = sum(client_sizes)
    return [
        sum((s / total) * w for w, s in zip(layer_ws, client_sizes))
        for layer_ws in zip(*client_weights)
    ]


def local_train(global_weights, x_c, y_c, local_epochs, batch_size, lr):
    m = build_model(learning_rate=lr)
    m.set_weights(global_weights)
    m.fit(x_c, y_c, epochs=local_epochs, batch_size=batch_size, verbose=0)
    return m.get_weights()


def run_fl_and_attack(client_data, member_idx, num_rounds, local_epochs,
                      batch_size, lr, n_attack=3000, seed=42):
    gc.collect()
    tf.keras.backend.clear_session()

    client_sizes = [len(xc) for xc, _ in client_data]
    gm = build_model(learning_rate=lr)
    gw = gm.get_weights()
    round_accs = []

    for r in range(num_rounds):
        cw_list = [local_train(gw, xc, yc, local_epochs, batch_size, lr)
                   for xc, yc in client_data]
        gw = fedavg_aggregate(cw_list, client_sizes)
        gm.set_weights(gw)
        _, acc = gm.evaluate(x_test, y_test, verbose=0)
        round_accs.append(acc)
        print(f'  Round {r+1:02d}/{num_rounds} | Acc: {acc:.4f}')

    # Build attack dataset
    rng   = np.random.default_rng(seed)
    n     = min(n_attack, len(member_idx), len(x_test))
    m_idx = rng.choice(member_idx,  size=n, replace=False)
    nm_idx= rng.choice(len(x_test), size=n, replace=False)

    m_probs  = gm.predict(x_train[m_idx],  batch_size=256, verbose=0)
    nm_probs = gm.predict(x_test[nm_idx],  batch_size=256, verbose=0)
    X_atk = np.vstack([m_probs, nm_probs])
    y_atk = np.array([1]*n + [0]*n)

    # Loss-based attack
    epsilon = 1e-10
    pred_classes = np.argmax(X_atk, axis=1)
    oh = np.zeros_like(X_atk)
    for i, c in enumerate(pred_classes):
        oh[i, c] = 1
    loss_scores = -(-np.sum(oh * np.log(X_atk + epsilon), axis=1))
    loss_auc    = roc_auc_score(y_atk, loss_scores)

    # Logistic regression attack
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_atk, y_atk, test_size=0.3, random_state=seed, stratify=y_atk)
    attacker = LogisticRegression(max_iter=1000, random_state=seed)
    attacker.fit(X_tr, y_tr)
    y_pred      = attacker.predict(X_te)
    y_pred_prob = attacker.predict_proba(X_te)[:, 1]
    fpr, tpr, _ = roc_curve(y_te, y_pred_prob)
    lr_auc      = roc_auc_score(y_te, y_pred_prob)
    lr_acc      = accuracy_score(y_te, y_pred)
    lr_recall   = recall_score(y_te, y_pred)

    del gm, cw_list, m_probs, nm_probs, X_atk
    gc.collect()

    return {
        'global_acc' : round_accs[-1],
        'round_accs' : round_accs,
        'lr_auc'     : lr_auc,
        'lr_acc'     : lr_acc,
        'lr_recall'  : lr_recall,
        'loss_auc'   : loss_auc,
        'fpr'        : fpr,
        'tpr'        : tpr,
    }


print('All functions defined.')


## 4. CIFAR-10 — IID Baseline


In [ ]:
print('=' * 55)
print('CIFAR-10 — IID Partitioning')
print('=' * 55)
iid_data, iid_idx = partition_data_iid(x_train, y_train, NUM_CLIENTS, RANDOM_SEED)
iid_members = [i for sub in iid_idx for i in sub]
for i, (xc, yc) in enumerate(iid_data):
    print(f'  Client {i+1}: {xc.shape[0]} samples | {len(np.unique(yc))} classes')

print('\nRunning FL + attack...')
iid_results = run_fl_and_attack(
    iid_data, iid_members,
    NUM_ROUNDS, LOCAL_EPOCHS, BATCH_SIZE, LEARNING_RATE
)
print(f'\n  Global accuracy : {iid_results["global_acc"]:.4f}')
print(f'  LR attack AUC   : {iid_results["lr_auc"]:.4f}')
print(f'  Loss attack AUC : {iid_results["loss_auc"]:.4f}')


## 5. CIFAR-10 — Non-IID


In [ ]:
print('=' * 55)
print(f'CIFAR-10 — Non-IID ({CLASSES_PER_CLIENT} classes per client)')
print('=' * 55)
noniid_data, noniid_idx = partition_data_noniid(
    x_train, y_train, NUM_CLIENTS,
    classes_per_client=CLASSES_PER_CLIENT,
    seed=RANDOM_SEED
)
noniid_members = [i for sub in noniid_idx for i in sub]

print('\nRunning FL + attack...')
noniid_results = run_fl_and_attack(
    noniid_data, noniid_members,
    NUM_ROUNDS, LOCAL_EPOCHS, BATCH_SIZE, LEARNING_RATE
)
print(f'\n  Global accuracy : {noniid_results["global_acc"]:.4f}')
print(f'  LR attack AUC   : {noniid_results["lr_auc"]:.4f}')
print(f'  Loss attack AUC : {noniid_results["loss_auc"]:.4f}')


## 6. IID vs Non-IID — Results & Plots


In [ ]:
print('═' * 60)
print('CIFAR-10 — IID vs Non-IID Comparison')
print('═' * 60)
print(f'{"Condition":<25} {"Global Acc":>12} {"LR AUC":>10} {"Loss AUC":>10}')
print('-' * 60)
print(f'{"IID":<25} {iid_results["global_acc"]:>12.4f} {iid_results["lr_auc"]:>10.4f} {iid_results["loss_auc"]:>10.4f}')
print(f'{"Non-IID (5 classes)":<25} {noniid_results["global_acc"]:>12.4f} {noniid_results["lr_auc"]:>10.4f} {noniid_results["loss_auc"]:>10.4f}')
print('═' * 60)
print(f'LR AUC increase  : +{noniid_results["lr_auc"] - iid_results["lr_auc"]:.4f}')
print(f'Loss AUC increase: +{noniid_results["loss_auc"] - iid_results["loss_auc"]:.4f}')
print('Random-chance baseline = 0.50')

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Convergence curves
ax = axes[0]
ax.plot(range(1, NUM_ROUNDS+1), iid_results['round_accs'],
        marker='o', linewidth=2, label='IID', color='steelblue')
ax.plot(range(1, NUM_ROUNDS+1), noniid_results['round_accs'],
        marker='s', linewidth=2, label='Non-IID', color='coral')
ax.set_xlabel('Round')
ax.set_ylabel('Test accuracy')
ax.set_title('FedAvg Convergence\nIID vs Non-IID')
ax.legend()
ax.grid(True, alpha=0.3)

# AUC comparison bar chart
ax = axes[1]
methods = ['LR\nIID', 'LR\nNon-IID', 'Loss\nIID', 'Loss\nNon-IID']
aucs    = [iid_results['lr_auc'], noniid_results['lr_auc'],
           iid_results['loss_auc'], noniid_results['loss_auc']]
colors  = ['#378ADD', '#0F6E56', '#378ADD', '#0F6E56']
bars = ax.bar(methods, aucs, color=colors, alpha=0.85)
ax.axhline(0.5, linestyle='--', color='gray', label='Random chance')
ax.set_ylabel('AUC')
ax.set_title('Attack AUC\nIID vs Non-IID')
ax.set_ylim(0.3, 1.0)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, aucs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# ROC curves
ax = axes[2]
ax.plot(iid_results['fpr'],    iid_results['tpr'],
        label=f'IID (AUC={iid_results["lr_auc"]:.3f})',
        color='steelblue', linewidth=2)
ax.plot(noniid_results['fpr'], noniid_results['tpr'],
        label=f'Non-IID (AUC={noniid_results["lr_auc"]:.3f})',
        color='coral', linewidth=2)
ax.plot([0,1],[0,1], '--', color='gray', linewidth=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — LR Attacker\nCIFAR-10')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('CIFAR-10 — IID vs Non-IID MIA Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cifar10_noniid_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()


## 7. Save & Download


In [ ]:
results = {
    'cifar10_iid'    : {k: float(v) if not isinstance(v, list) else v
                        for k, v in iid_results.items() if k not in ['fpr','tpr']},
    'cifar10_noniid' : {k: float(v) if not isinstance(v, list) else v
                        for k, v in noniid_results.items() if k not in ['fpr','tpr']},
}
with open(os.path.join(OUTPUT_DIR, 'cifar10_noniid_results.json'), 'w') as f:
    json.dump(results, f, indent=2)

from google.colab import files
files.download(os.path.join(OUTPUT_DIR, 'cifar10_noniid_results.json'))
files.download(os.path.join(OUTPUT_DIR, 'cifar10_noniid_comparison.png'))
print('Done. Run Notebook 03b next for CIFAR-100.')
